Flora HUANG

# Tableau 1 : Caractéristiques démographiques et cliniques de base

Consigne : Créer une table descriptive présentant les caractéristiques de la population étudiée à l'inclusion (baseline), stratifiée par bras de traitement.

Cette table doit inclure :

- Variable démographique (âge, sexe, origine ethnique, IMC)
- Antécédents médicaux pertinents
- Paramètres cliniques et biologiques de base
- Statistiques appropriées (moyenne ± écart-type, médiane [IQR], effectifs et pourcentages)

In [57]:
#Importer les données nécessaire
import polars as pl # pour manipulation de données
import pandas as pd # pour TableOne

Avant de commencer la création du table 1, l'exploration des contenus des fichiers est nécessaire : 
- Table Excel : VariableTable1 : variable qui vont potentiellement nous intéresser :
1) AMDISSION : Patient hospitalise et ambulatoire ( in-patient / out-patient)
2) GENDRE : Sexe (F/M)
3) AGE : Age (année)
4) ARM : Bras de traitement ( BUPRENORPHINE-NALOXONE / CLONIDINE /  SCREEN FAILURE)
5) HEIGHT : Taille (m)
6) WEIGHT : Poids (kg)
7) SBP : Pression systolique (mmHg)
8) DBP : Pression diastolique (mmHg)
9) PULSE : Frequence cardiaque (bpm)
10) REST : Frequence respiratoire (/min)
11) TEMP : Temperature corporelle (C°)
12) METHADONE.sc : Test urinaire methadone - debut etude (+/-)
13) MORPHINE.sc : Test urinaire morphine - debut etude (+/-)
14) IMC : n'est pas dans la table excel -> à ajouter dans la table de résultat)

Il est possible de les retrouver dans les fichiers suivant : 
-  dm.csv : données démographiques
- vs.csv : signes vitaux
- mh.csv : antécédents médicaux
- lb.csv : tests urinaires

# Données démographiques : sexe / age / origine ethnique / IMC

In [2]:
# Charger les données des fichiers 'DM'
dm1 = pl.read_csv("dm1.csv")
dm2 = pl.read_csv('dm2.csv')
dm = pl.concat([dm1, dm2], how="vertical")

In [3]:
#Aperçu du fichier
dm.shape

(411, 20)

Le nombre de participants inclus à la baseline pour établir notre tableau descriptif est de 411 sujets.

In [4]:
dm.columns

['STUDYID',
 'DOMAIN',
 'USUBJID',
 'EPOCH',
 'VISIT',
 'VISITNUM',
 'RFSTDTC',
 'RFENDTC',
 'SITEID',
 'BRTHDTC',
 'AGE',
 'AGEU',
 'SEX',
 'RACE',
 'ETHNIC',
 'ARMCD',
 'ARM',
 'COUNTRY',
 'DMDTC',
 'DMDY']

In [5]:
dm.sample(4)

STUDYID,DOMAIN,USUBJID,EPOCH,VISIT,VISITNUM,RFSTDTC,RFENDTC,SITEID,BRTHDTC,AGE,AGEU,SEX,RACE,ETHNIC,ARMCD,ARM,COUNTRY,DMDTC,DMDY
str,str,str,str,str,i64,i64,i64,str,i64,str,str,str,str,str,str,str,str,i64,str
"""NIDA-CTN-0001""","""DM""","""01_096713""","""SCREENING""","""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,2001,null,1957,"""44.197125257""","""YEARS""","""M""","""BLACK, AFRICAN AMERICAN, OR NE…",null,"""BUPNAL""","""BUPRENORPHINE/NALOXONE""","""USA""",2001,"""1"""
"""NIDA-CTN-0002""","""DM""","""02_071895""","""SCREENING""","""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,2001,null,1967,"""34.236824093""","""YEARS""","""M""","""SPANISH, HISPANIC, OR LATINO""","""HISPANIC""","""BUPNAL""","""BUPRENORPHINE/NALOXONE""","""USA""",2001,"""1"""
"""NIDA-CTN-0002""","""DM""","""02_029311""","""SCREENING""","""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,2001,null,1980,"""20.97467488""","""YEARS""","""M""","""WHITE""",null,"""BUPNAL""","""BUPRENORPHINE/NALOXONE""","""USA""",2001,"""-1"""
"""NIDA-CTN-0001""","""DM""","""01_049581""","""SCREENING""","""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,2002,null,1965,"""36.709103354""","""YEARS""","""M""","""SPANISH, HISPANIC, OR LATINO""","""HISPANIC""","""BUPNAL""","""BUPRENORPHINE/NALOXONE""","""USA""",2001,"""-1"""


In [6]:
# Sélectionner et convertir les colonnes pertinant pour notre table 
dm = dm.select(
    pl.col('USUBJID'), #Identifiant 
    pl.col('AGE'),
    pl.col('SEX'),
    pl.col('ETHNIC'),
    pl.col('ARMCD'),
)
dm

USUBJID,AGE,SEX,ETHNIC,ARMCD
str,str,str,str,str
"""01_000579""","""27.282683094""","""F""",null,"""CLON"""
"""01_001362""","""41.429158111""","""F""",null,"""CLON"""
"""01_001490""","""30.392881588""","""M""",null,"""BUPNAL"""
"""01_002199""","""20.928131417""","""M""",null,"""BUPNAL"""
"""01_002844""","""19.775496235""","""F""",null,"""BUPNAL"""
…,…,…,…,…
"""02_098425""","""49.221081451""","""M""",null,"""CLON"""
"""02_098689""",""" ""","""M""","""HISPANIC""","""SCRFAIL"""
"""02_099053""","""43.945242984""","""M""",null,"""BUPNAL"""


On observe que la colonne AGE est en "str", il est essentiel de la convertir en float pour obtenir des valeurs numériques exploitables pour les analyses statistiques

In [7]:
dm = dm.select(
    pl.col('USUBJID'), 
    pl.col('AGE').cast(pl.Float64),
    pl.col('SEX'),
    pl.col('ETHNIC'),
    pl.col('ARMCD'),
)
dm

InvalidOperationError: conversion from `str` to `f64` failed in column 'AGE' for 3 out of 17 values: [" ", " ", " "]

Il est inscrit que nous retrouvons des valeurs manquantes dans la colonne 'AGE', la convertion n'est donc pas possible. 
Pour cela un nettoyage des valeurs manquantes ou incorrectes est nécessaire.

In [8]:
# Nettoyage des valeurs manquantes ou incorrectes : nous gardons que les sujets ayant un valeur > 3
dm = dm.filter(pl.col('AGE').str.len_chars() > 3)

In [9]:
# Conversion AGE → float
dm = dm.select(
    pl.col('USUBJID'),
    pl.col('AGE').cast(pl.Float64),
    pl.col('SEX'),
    pl.col('ETHNIC'),
    pl.col('ARMCD'),
)
dm

USUBJID,AGE,SEX,ETHNIC,ARMCD
str,f64,str,str,str
"""01_000579""",27.282683,"""F""",null,"""CLON"""
"""01_001362""",41.429158,"""F""",null,"""CLON"""
"""01_001490""",30.392882,"""M""",null,"""BUPNAL"""
"""01_002199""",20.928131,"""M""",null,"""BUPNAL"""
"""01_002844""",19.775496,"""F""",null,"""BUPNAL"""
…,…,…,…,…
"""02_098074""",50.176591,"""M""",null,"""BUPNAL"""
"""02_098425""",49.221081,"""M""",null,"""CLON"""
"""02_099053""",43.945243,"""M""",null,"""BUPNAL"""


In [10]:
# Valeurs uniques de la colonne'ARM'
dm["ARMCD"].value_counts()

ARMCD,count
str,u32
"""BUPNAL""",233
"""CLON""",109


In [11]:
# Valeurs uniques de la colonne'SEX'
dm["SEX"].value_counts()

SEX,count
str,u32
"""F""",110
"""M""",232


In [12]:
# Valeurs uniques de la colonne'ETHNIC'
dm["ETHNIC"].value_counts()

ETHNIC,count
str,u32
"""HISPANIC""",69
null,273


# Chargement et traitement des données VS (Vital Signs).
Nous retrouvons les valeurs de la taille et du poids qui nous permettrons de calculer l'IMC (ou BMI) et les sigaux vitaux des sujets.


In [14]:
# Charger les fichiers VS1 et VS2
vs1 = pl.read_csv("vs1.csv")
vs2 = pl.read_csv('vs2.csv')

# Concaténer les fichiers
vs = pl.concat([vs1, vs2], how="vertical")

In [15]:
#Aperçu du fichier
vs.shape

(18994, 19)

In [16]:
vs.columns

['STUDYID',
 'DOMAIN',
 'USUBJID',
 'EPOCH',
 'VSSEQ',
 'VSTESTCD',
 'VSTEST',
 'VSCAT',
 'VSPOS',
 'VSORRES',
 'VSORRESU',
 'VSSTRESC',
 'VSSTRESN',
 'VSSTRESU',
 'VSBLFL',
 'VISIT',
 'VISITNUM',
 'VSDTC',
 'VSDY']

In [17]:
vs.sample(4)

STUDYID,DOMAIN,USUBJID,EPOCH,VSSEQ,VSTESTCD,VSTEST,VSCAT,VSPOS,VSORRES,VSORRESU,VSSTRESC,VSSTRESN,VSSTRESU,VSBLFL,VISIT,VISITNUM,VSDTC,VSDY
str,str,str,str,i64,str,str,str,str,f64,str,f64,f64,str,str,str,i64,i64,i64
"""NIDA-CTN-0001""","""VS""","""01_024957""","""ACTIVE""",85,"""SBP""","""SYSTOLIC BLOOD PRESSURE""","""VITAL SIGNS FORM""","""SITTING""",120.0,"""MMHG""",120.0,120.0,"""MMHG""",null,"""STUDY DAY 10""",10,2001,10
"""NIDA-CTN-0001""","""VS""","""01_047446""","""ACTIVE""",55,"""RESP""","""RESPIRATIONS""","""VITAL SIGNS FORM""","""SITTING""",16.0,"""BREATHS/MIN""",16.0,16.0,"""BREATHS/MIN""",null,"""STUDY DAY 8""",8,2001,8
"""NIDA-CTN-0001""","""VS""","""01_006566""","""ACTIVE""",36,"""PULSE""","""PULSE""","""VITAL SIGNS FORM""","""SITTING""",88.0,"""BEATS/MINUTE""",88.0,88.0,"""BEATS/MINUTE""",null,"""STUDY DAY 5""",5,2001,5
"""NIDA-CTN-0001""","""VS""","""01_073890""","""ACTIVE""",69,"""TEMP""","""TEMPERATURE""","""VITAL SIGNS FORM""","""SITTING""",98.8,"""F""",98.8,98.8,"""F""",null,"""STUDY DAY 11""",11,2001,11


In [18]:
# Sélection des colonnes pertinant pour notre table 
vs = vs.select(
    pl.col('USUBJID'),  # identifiant
    pl.col('VSORRES'),  # valeurs numérique des signes vitaux
    pl.col('VSORRESU'),  # unités qui les corresponds
    pl.col('VSTESTCD'),  # permet de différencier entre l'unité mmHg de la Pression systolique et de la Pression diastolyque pression diastolyque
)
vs

USUBJID,VSORRES,VSORRESU,VSTESTCD
str,f64,str,str
"""01_000579""",80.0,"""MMHG""","""DBP"""
"""01_000579""",62.0,"""MMHG""","""DBP"""
"""01_000579""",50.0,"""MMHG""","""DBP"""
"""01_000579""",52.0,"""MMHG""","""DBP"""
"""01_000579""",67.0,"""INCHES""","""HEIGHT"""
…,…,…,…
"""02_099926""",106.0,"""MMHG""","""SBP"""
"""02_099926""",98.6,"""F""","""TEMP"""
"""02_099926""",96.8,"""F""","""TEMP"""


In [19]:
# Valeurs uniques de la colonne'VSORRESU'
vs["VSORRESU"].value_counts()

VSORRESU,count
str,u32
"""MMHG""",6058
"""INCHES""",377
"""F""",2936
null,11
"""BREATHS/MIN""",2954
"""POUNDS""",378
"""BEATS/MINUTE""",6280


Dans les fichiers CNT, les unités choisies pour la taille et le poids ne sont pas précisées (inches or centimeters/pounds or kilograms). 

Pour les identifier, la fonction value_counts() permet d’observer les différentes valeurs possibles dans la colonne VSORRESU.

Nous constatons ainsi que seules les unités suivantes sont utilisées :
- Poids : POUNDS
- Taille : INCHES

Nous remarquons aussi les unités des signes vitaux suivants : 
- F : pour degré Fahrenheit
- Breaths/min : pour fréquence respiratoire
- mmHg (millimètres de mercure) : pour la pression diastolyique et systolique 

In [20]:
# Valeurs uniques de la colonne'VSTESTCD' 
vs["VSTESTCD"].value_counts()

VSTESTCD,count
str,u32
"""TEMP""",2941
"""DBP""",3029
"""HEIGHT""",380
"""SBP""",3029
"""WEIGHT""",381
"""PULSE""",6280
"""RESP""",2954


Avec value_counts(), comme pour la colonne précédente, nous pouvons observer dans la colonne 'VSTESTCD', les différents valeurs des signes vitaux des sujets

Par la suite, pour les valeurs provenant du fichier vs.csv, nous créons des DataFrames séparés pour chaque mesure afin d’en faciliter l’intégration dans notre Table 1.

In [21]:
# POIDS (POUNDS)
weight = vs.filter(pl.col("VSORRESU") == "POUNDS").select([
    pl.col("USUBJID"),
    pl.col("VSORRES").alias("Weight_lbs") # le changement du nom de la valeur permet une meilleure comphréhension 
]).unique(subset="USUBJID")
weight

USUBJID,Weight_lbs
str,f64
"""02_093746""",210.0
"""02_074773""",152.0
"""02_059257""",187.0
"""01_045453""",164.0
"""02_040804""",175.0
…,…
"""01_002844""",116.0
"""02_039697""",180.0
"""02_008740""",112.0


In [22]:
# TAILLE (INCHES)
height = vs.filter(pl.col("VSORRESU") == "INCHES").select([
    pl.col("USUBJID"),
    pl.col("VSORRES").alias("Height_inches"),
]).unique(subset="USUBJID")
height

USUBJID,Height_inches
str,f64
"""02_004450""",72.0
"""02_087712""",78.0
"""01_050792""",74.0
"""02_081773""",55.0
"""01_021516""",73.0
…,…
"""01_012272""",53.0
"""02_023773""",63.0
"""02_091644""",75.0


In [23]:
# PRESSION SYSTOLIQUE (mmHg)
sbp = vs.filter(pl.col("VSTESTCD") == "SBP").select([
    pl.col("USUBJID"),
    pl.col("VSORRES").alias("Systolic_BP"),
]).unique(subset="USUBJID")
sbp

USUBJID,Systolic_BP
str,f64
"""01_009835""",140.0
"""02_027910""",119.0
"""02_008740""",107.0
"""02_032208""",110.0
"""01_097261""",116.0
…,…
"""02_044005""",113.0
"""02_013991""",138.0
"""01_027452""",130.0


In [24]:
# PRESSION DIASTOLIQUE (mmHg)
dbp = vs.filter(pl.col("VSTESTCD") == "DBP").select([
    pl.col("USUBJID"),
    pl.col("VSORRES").alias("Diastolic_BP"),
]).unique(subset="USUBJID")
dbp

USUBJID,Diastolic_BP
str,f64
"""02_080565""",70.0
"""02_032871""",72.0
"""02_027772""",68.0
"""02_044380""",63.0
"""01_037378""",64.0
…,…
"""01_006180""",60.0
"""02_050467""",78.0
"""01_093422""",80.0


In [25]:
# FRÉQUENCE CARDIAQUE (bpm)
pulse = vs.filter(pl.col("VSTESTCD") == "PULSE").select([
    pl.col("USUBJID"),
    pl.col("VSORRES").alias("Heart_Rate"),
]).unique(subset="USUBJID")
pulse

USUBJID,Heart_Rate
str,f64
"""02_007189""",80.0
"""02_013342""",85.0
"""02_048454""",80.0
"""02_000852""",61.0
"""02_055979""",60.0
…,…
"""01_093422""",84.0
"""01_086495""",84.0
"""02_009190""",92.0


In [26]:
# TEMPÉRATURE (°F : 1C° = 33.8°F)
temp = vs.filter(pl.col("VSTESTCD") == "TEMP").select([
    pl.col("USUBJID"),
    pl.col("VSORRES").alias("Temperature_F"),
]).unique(subset="USUBJID")
temp

USUBJID,Temperature_F
str,f64
"""01_097597""",98.8
"""01_047430""",97.1
"""02_037401""",99.2
"""02_043130""",98.4
"""02_052516""",98.6
…,…
"""02_063128""",99.2
"""02_058079""",97.0
"""02_097175""",99.8


Calcule du BMI : colonnes qui nous intéressent : 
- USUBJID : identifiant -> il nous permet de faire une jointure avec le fichier 'dm.csv' 
- VSORRES : pour height / weight 
- VSORRESU : pour leurs unités (inches/pounds)

-> Ces valeurs nous permettra de calculer l'IMC des sujets : 
- IMC =  poids / (taille)²
- ou : BMI = 703 × weight (pounds) / [height (inches)]²

In [27]:
# BMI
# Jointure entre les valeurs'POUNDS' et 'INCHES' de la colonne'VSORRES'
bmi = weight.join(height, on="USUBJID", how="inner")

# Calcule du BMI : BMI = 703 × weight (lb) / [height (in)]²
bmi = bmi.with_columns(
    (703 * pl.col("Weight_lbs") / (pl.col("Height_inches") ** 2)).alias("BMI")
)
bmi

USUBJID,Weight_lbs,Height_inches,BMI
str,f64,f64,f64
"""02_093746""",210.0,66.0,33.891185
"""02_074773""",152.0,68.0,23.108997
"""02_059257""",187.0,75.0,23.370844
"""01_045453""",164.0,70.0,23.52898
"""02_040804""",175.0,68.0,26.605753
…,…,…,…
"""01_002844""",116.0,60.0,22.652222
"""02_039697""",180.0,68.0,27.365917
"""02_008740""",112.0,60.0,21.871111


Pour les fichiers 'vs.csv', nous observons donc les dataframes suivants : 
- Température : fahrenheit (F) : temp
- Fréquence cardiaque (bpm) : pulse
- Pression diastolique (mmHg) : dbp
- pression systolique (mmHg) : sbp
- taille (lbs) : height
- poids (pouds) : weight
- BMI : bmi

# Antécédents médicaux 
Nous explorons les fichiers 'mh.csv', afin d'ajouter les valeurs des antécédents médicaux des sujets incluses.

In [28]:
mh1 = pl.read_csv("mh1.csv")
mh2 = pl.read_csv('mh2.csv')
mh = pl.concat([mh1, mh2], how="vertical")

In [29]:
mh.columns

['STUDYID',
 'DOMAIN',
 'USUBJID',
 'EPOCH',
 'MHSEQ',
 'MHSPID',
 'MHTERM',
 'MHOCCUR',
 'MHSTAT',
 'VISIT',
 'VISITNUM',
 'MHDTC',
 'MHDY',
 'MHENRF']

In [30]:
mh.shape

(12412, 14)

In [31]:
mh.sample(4)

STUDYID,DOMAIN,USUBJID,EPOCH,MHSEQ,MHSPID,MHTERM,MHOCCUR,MHSTAT,VISIT,VISITNUM,MHDTC,MHDY,MHENRF
str,str,str,str,i64,i64,str,str,str,str,i64,i64,i64,str
"""NIDA-CTN-0002""","""MH""","""02_045563""","""SCREENING""",5,3,"""CARDIOVASCULAR""","""Y""",null,"""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,-7,"""BEFORE"""
"""NIDA-CTN-0001""","""MH""","""01_038734""","""SCREENING""",18,9,"""GASTROINTESTINAL""","""N""",null,"""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,null,"""DURING/AFTER"""
"""NIDA-CTN-0002""","""MH""","""02_046737""","""SCREENING""",7,4,"""RESPIRATORY""","""N""",null,"""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,1,"""BEFORE"""
"""NIDA-CTN-0002""","""MH""","""02_044005""","""SCREENING""",32,16,"""ALLERGIES""","""N""",null,"""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,1,"""DURING/AFTER"""


In [32]:
# Sélection des colonnes pertinant pour notre table dans le fichier 'mh.csv'
mh = mh.select(
    pl.col('USUBJID'), # identifiant
    pl.col('MHTERM'), # les types de maladies
    pl.col('MHENRF'), # période de la contraction de la maladie -> On s'intéresse seulement à la valeur 'BEFORE' car on veut les antécédents médicaux
    pl.col('MHOCCUR'), # On s'intéresse seulement à la valeur 'Y' pour yes -> renvoie à la colonne 'MHENRF', pour nous c'est le cas 'BEFORE' 
)
mh

USUBJID,MHTERM,MHENRF,MHOCCUR
str,str,str,str
"""01_000579""","""ALLERGIES""","""BEFORE""","""N"""
"""01_000579""","""ALLERGIES""","""DURING/AFTER""","""N"""
"""01_000579""","""CARDIOVASCULAR""","""BEFORE""","""N"""
"""01_000579""","""CARDIOVASCULAR""","""DURING/AFTER""","""N"""
"""01_000579""","""DERMATOLOGICAL""","""BEFORE""","""N"""
…,…,…,…
"""02_099926""","""SYMPTOMS OF TB""","""DURING/AFTER""","""N"""
"""02_099926""","""TB SKIN TEST POSITIVE""","""BEFORE""","""Y"""
"""02_099926""","""TB SKIN TEST POSITIVE""","""DURING/AFTER""","""N"""


Nous filtrons uniquement les valeurs suivantes pour obtenir les antécédents médicaux:

- Condition 1 : MHENRF = 'BEFORE' (antécédents avant l'étude)
- Condition 2 : MHOCCUR = 'Y' (antécédents présents)

In [33]:
# Faire un filtrage des antécédents (BEFORE + Y)
mh_filtered = mh.filter(
    (pl.col("MHENRF") == "BEFORE") & 
    (pl.col("MHOCCUR") == "Y")
)
mh_filtered

USUBJID,MHTERM,MHENRF,MHOCCUR
str,str,str,str
"""01_000579""","""GENITOURINARY""","""BEFORE""","""Y"""
"""01_000579""","""NEUROLOGICAL""","""BEFORE""","""Y"""
"""01_000579""","""OTHER""","""BEFORE""","""Y"""
"""01_000579""","""PSYCHIATRIC""","""BEFORE""","""Y"""
"""01_000579""","""SEIZURE""","""BEFORE""","""Y"""
…,…,…,…
"""02_098425""","""GASTROINTESTINAL""","""BEFORE""","""Y"""
"""02_099926""","""ALLERGIES""","""BEFORE""","""Y"""
"""02_099926""","""OTHER""","""BEFORE""","""Y"""


In [34]:
# Pour chaque patient on compte le nombre d'antécédents de maladie 
mh_count = mh_filtered.group_by("USUBJID").agg(
    pl.count("MHTERM").alias("MH_Count")
)
mh_count 

USUBJID,MH_Count
str,u32
"""02_051268""",2
"""01_033375""",7
"""01_055899""",3
"""02_008177""",2
"""02_047605""",5
…,…
"""02_069875""",2
"""02_083268""",2
"""02_048792""",3


# DONNÉES BIOLOGIQUES (LB) : 
Nous explorons les fichiers "lb.csv", afin d'observer la présence ou non des subtances : méthadone et morphine

In [35]:
lb1 = pl.read_csv('lb1.csv')
lb2 = pl.read_csv('lb2.csv')
lb = pl.concat([lb1, lb2], how="vertical")

In [36]:
#Vérification 
lb.sample(4)

STUDYID,DOMAIN,USUBJID,EPOCH,LBSEQ,LBTESTCD,LBTEST,LBCAT,LBORRES,LBORRESU,LBORNRLO,LBORNRHI,LBSTRESC,LBSTRESN,LBSTRESU,LBSTNRLO,LBSTNRHI,LBNRIND,LBSTAT,LBREASND,LBSPEC,LBMETHOD,LBBLFL,VISIT,VISITNUM,LBDTC,LBDY
str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str
"""NIDA-CTN-0001""","""LB""","""01_090085""","""ACTIVE""",25,"""AMPHET""","""AMPHETAMINES""","""DRUG SCREEN""","""NEGATIVE""",null,null,null,"""NEGATIVE""",""" """,null,""" """,""" """,null,null,null,"""URINE""","""CENTRAL LAB""",null,"""STUDY DAY 7""",7,2001,"""7"""
"""NIDA-CTN-0001""","""LB""","""01_096723""","""ACTIVE""",35,"""BENZOD""","""BENZODIAZEPINES""","""DRUG SCREEN""","""NEGATIVE""",null,null,null,"""NEGATIVE""",""" """,null,""" """,""" """,null,null,null,"""URINE""","""ACCUTEST""",null,"""STUDY DAY 14""",14,2001,"""14"""
"""NIDA-CTN-0002""","""LB""","""02_048107""","""ACTIVE""",18,"""THC""","""THC""","""DRUG SCREEN""","""NEGATIVE""",null,null,null,"""NEGATIVE""",""" """,null,""" """,""" """,null,null,null,"""URINE""","""CENTRAL LAB""",null,"""STUDY DAY 3""",3,2001,"""3"""
"""NIDA-CTN-0002""","""LB""","""02_061221""","""ACTIVE""",49,"""AMPHET""","""AMPHETAMINES""","""DRUG SCREEN""","""NEGATIVE""",null,null,null,"""NEGATIVE""",""" """,null,""" """,""" """,null,null,null,"""URINE""","""CENTRAL LAB""",null,"""STUDY DAY 14""",14,2001,"""14"""


In [37]:
lb.columns

['STUDYID',
 'DOMAIN',
 'USUBJID',
 'EPOCH',
 'LBSEQ',
 'LBTESTCD',
 'LBTEST',
 'LBCAT',
 'LBORRES',
 'LBORRESU',
 'LBORNRLO',
 'LBORNRHI',
 'LBSTRESC',
 'LBSTRESN',
 'LBSTRESU',
 'LBSTNRLO',
 'LBSTNRHI',
 'LBNRIND',
 'LBSTAT',
 'LBREASND',
 'LBSPEC',
 'LBMETHOD',
 'LBBLFL',
 'VISIT',
 'VISITNUM',
 'LBDTC',
 'LBDY']

In [38]:
lb.shape

(17977, 27)

Les colonnes qui nous intéressant sont les suivants : 
- USUBJID : identifiant
- LBTEST : test pour détecter la présence de substance : ici Morphine et Méthadone 
- LBORRES : résultats : positive/négativ

In [39]:
# Sélection des colonnes pertinant pour notre table dans le fichier 'mh.csv'
lb = lb.select(
    pl.col('USUBJID'), 
    pl.col('LBTEST'), 
    pl.col('LBORRES'), 
)
lb

USUBJID,LBTEST,LBORRES
str,str,str
"""01_000579""","""AMPHETAMINES""","""NEGATIVE"""
"""01_000579""","""AMPHETAMINES""","""NEGATIVE"""
"""01_000579""","""BARBITURATES""","""POSITIVE"""
"""01_000579""","""BENZODIAZEPINES""","""POSITIVE"""
"""01_000579""","""BENZODIAZEPINES""","""NEGATIVE"""
…,…,…
"""02_099926""","""THC""","""NEGATIVE"""
"""02_099926""","""THC""","""NEGATIVE"""
"""02_099926""","""THC""","""NEGATIVE"""


In [40]:
lb["LBTEST"].unique()

LBTEST
str
"""TCA"""
"""METHADONE"""
"""NONE"""
"""OPIATE"""
"""AMPHETAMINES"""
…
"""MORPHINE"""
"""BENZODIAZEPINES"""
"""METHAMPHETAMINE"""


Nous séparons la valeurs 'Morphine' et 'Méthadone' de la colonne 'LBTEST' en 2 dataframe afin de pouvoir les distinguer.

In [41]:
# MORPHINE
morphine_test = lb.filter(
    pl.col("LBTEST").str.contains("MORPHINE") 
).select([
    "USUBJID",
    pl.col("LBORRES").alias("Morphine_Test")
]).unique(subset="USUBJID")
morphine_test

USUBJID,Morphine_Test
str,str
"""02_044005""","""POSITIVE"""
"""01_065758""","""POSITIVE"""
"""02_075531""","""POSITIVE"""
"""02_021273""","""NEGATIVE"""
"""02_092156""","""POSITIVE"""
…,…
"""01_053466""","""POSITIVE"""
"""02_064002""","""POSITIVE"""
"""02_040896""","""POSITIVE"""


In [42]:
# MÉTHADONE
methadone_test = lb.filter(
    pl.col("LBTEST").str.contains("METHADONE") 
).select([
    "USUBJID",
    pl.col("LBORRES").alias("Methadone_Test")
]).unique(subset="USUBJID")
methadone_test

USUBJID,Methadone_Test
str,str
"""02_099926""","""NEGATIVE"""
"""02_065163""","""NEGATIVE"""
"""02_040896""","""NEGATIVE"""
"""02_032208""","""NEGATIVE"""
"""02_097455""","""NEGATIVE"""
…,…
"""02_045101""","""NEGATIVE"""
"""02_095071""","""NEGATIVE"""
"""02_083701""","""NEGATIVE"""


# Fusion des données
Nous avons déjà créé les DataFrames suivant:
- dm (démographie)
- weight, height (signes vitaux)
- mh_count (antécédents)
- morphine_test, methadone_test (biologie)

In [44]:
# Nous commencons par la fusion avec le fichier 'dm.csv'
df_final = dm

In [45]:
# Nous rajoutons par la suite les signes vitaux
df_final = df_final.join(weight, on="USUBJID", how="left")
df_final = df_final.join(height, on="USUBJID", how="left")
df_final = df_final.join(bmi, on="USUBJID", how="left")
df_final = df_final.join(sbp, on="USUBJID", how="left")
df_final = df_final.join(dbp, on="USUBJID", how="left")
df_final = df_final.join(pulse, on="USUBJID", how="left")
df_final = df_final.join(temp, on="USUBJID", how="left")

In [46]:
# Puis les antécédents médicaux + test
df_final = df_final.join(mh_count, on="USUBJID", how="left")

In [47]:
df_final

USUBJID,AGE,SEX,ETHNIC,ARMCD,Weight_lbs,Height_inches,Weight_lbs_right,Height_inches_right,BMI,Systolic_BP,Diastolic_BP,Heart_Rate,Temperature_F,MH_Count
str,f64,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
"""01_000579""",27.282683,"""F""",null,"""CLON""",130.0,67.0,130.0,67.0,20.358654,132.0,80.0,72.0,98.8,5
"""01_001362""",41.429158,"""F""",null,"""CLON""",179.0,58.0,179.0,58.0,37.406956,114.0,76.0,72.0,98.0,null
"""01_001490""",30.392882,"""M""",null,"""BUPNAL""",155.0,72.0,155.0,72.0,21.019483,104.0,60.0,66.0,99.2,null
"""01_002199""",20.928131,"""M""",null,"""BUPNAL""",null,null,null,null,null,125.0,77.0,100.0,98.0,5
"""01_002844""",19.775496,"""F""",null,"""BUPNAL""",116.0,60.0,116.0,60.0,22.652222,100.0,80.0,92.0,97.8,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""02_098074""",50.176591,"""M""",null,"""BUPNAL""",132.0,62.0,132.0,62.0,24.140479,110.0,80.0,72.0,97.8,4
"""02_098425""",49.221081,"""M""",null,"""CLON""",250.0,71.0,250.0,71.0,34.864114,152.0,105.0,87.0,99.0,3
"""02_099053""",43.945243,"""M""",null,"""BUPNAL""",135.0,67.0,135.0,67.0,21.14168,120.0,82.0,80.0,99.4,null


In [48]:
# On ajout les tests urinaires
df_final = df_final.join(morphine_test, on="USUBJID", how="left")
df_final = df_final.join(methadone_test, on="USUBJID", how="left")

In [50]:
# On remplace les valeurs manquantes
df_final = df_final.with_columns([
    pl.col("MH_Count").fill_null(0),
    pl.col("Morphine_Test").fill_null("Non testé"),
    pl.col("Methadone_Test").fill_null("Non testé")
])

In [51]:
df_final

USUBJID,AGE,SEX,ETHNIC,ARMCD,Weight_lbs,Height_inches,Weight_lbs_right,Height_inches_right,BMI,Systolic_BP,Diastolic_BP,Heart_Rate,Temperature_F,MH_Count,Morphine_Test,Methadone_Test
str,f64,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,str,str
"""01_000579""",27.282683,"""F""",null,"""CLON""",130.0,67.0,130.0,67.0,20.358654,132.0,80.0,72.0,98.8,5,"""POSITIVE""","""NEGATIVE"""
"""01_001362""",41.429158,"""F""",null,"""CLON""",179.0,58.0,179.0,58.0,37.406956,114.0,76.0,72.0,98.0,0,"""NEGATIVE""","""POSITIVE"""
"""01_001490""",30.392882,"""M""",null,"""BUPNAL""",155.0,72.0,155.0,72.0,21.019483,104.0,60.0,66.0,99.2,0,"""POSITIVE""","""NEGATIVE"""
"""01_002199""",20.928131,"""M""",null,"""BUPNAL""",null,null,null,null,null,125.0,77.0,100.0,98.0,5,"""NEGATIVE""","""NEGATIVE"""
"""01_002844""",19.775496,"""F""",null,"""BUPNAL""",116.0,60.0,116.0,60.0,22.652222,100.0,80.0,92.0,97.8,0,"""POSITIVE""","""NEGATIVE"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""02_098074""",50.176591,"""M""",null,"""BUPNAL""",132.0,62.0,132.0,62.0,24.140479,110.0,80.0,72.0,97.8,4,"""POSITIVE""","""NEGATIVE"""
"""02_098425""",49.221081,"""M""",null,"""CLON""",250.0,71.0,250.0,71.0,34.864114,152.0,105.0,87.0,99.0,3,"""POSITIVE""","""NEGATIVE"""
"""02_099053""",43.945243,"""M""",null,"""BUPNAL""",135.0,67.0,135.0,67.0,21.14168,120.0,82.0,80.0,99.4,0,"""POSITIVE""","""NEGATIVE"""


In [52]:
# On convertit les données en Pandas pour pouvoir appliquer TableOne
df_pandas = df_final.to_pandas()

# Création de la Table1

In [53]:
# Variables catégorielles
cat_vars = [
    'SEX',                      # Sexe
    'ETHNIC',                   # Origine ethnique
    'Morphine_Test',            # Test morphine
    'Methadone_Test'            # Test méthadone
]

In [70]:
# Variables continues
cont_vars = [
    'AGE',                      # Âge
    'BMI',                      # BMI
    'Weight_lbs',               # Poids en livres
    'Height_inches',            # Tailles en pouces
    'Systolic_BP',              # Pression systolique
    'Diastolic_BP',             # Pression diastolique
    'Heart_Rate',               # Fréquence cardiaque
    'Temperature_F',            # Température en F°
    'MH_Count',                 # Nombre antécédents
]

In [71]:
# Variables non-normales (médiane)
non_norm = ['MH_Count']

In [72]:
from tableone import TableOne

In [73]:
# Créer la Table 1
table1 = TableOne(
    df_pandas,
    columns=cat_vars + cont_vars,
    categorical=cat_vars,
    continuous=cont_vars,
    nonnormal=non_norm,
    groupby='ARMCD',
    missing=False,     
    overall=False # Ne pas prendre en compte la colonne Overall car on ne prend en compte que les sujets incluses
)
table1

Grouped by ARMCD               
                                            BUPNAL           CLON
n                                              233            109
SEX, n (%)               F               72 (30.9)      38 (34.9)
                         M              161 (69.1)      71 (65.1)
ETHNIC, n (%)            HISPANIC        50 (21.5)      19 (17.4)
                         None           183 (78.5)      90 (82.6)
Morphine_Test, n (%)     NEGATIVE        67 (28.8)      28 (25.7)
                         POSITIVE       166 (71.2)      81 (74.3)
Methadone_Test, n (%)    NEGATIVE       212 (91.0)      99 (90.8)
                         POSITIVE         21 (9.0)       10 (9.2)
AGE, mean (SD)                         37.4 (10.5)     39.3 (9.3)
BMI, mean (SD)                          24.8 (5.0)     25.6 (4.6)
Weight_lbs, mean (SD)                 165.7 (35.4)   168.0 (35.5)
Height_inches, mean (SD)               71.2 (43.0)     67.8 (4.5)
Systolic_BP, mean (SD)                120.3 (15.8)   124.8 (17.6)
Diastolic_BP, mean (SD)                80.9 (42.6)    79.9 (11.6)
Heart_Rate, mean (SD)                  76.5 (11.7)    75.5 (12.0)
Temperature_F, mean (SD)              102.2 (59.1)     98.3 (0.8)
MH_Count, median [Q1,Q3]             2.0 [0.0,3.0]  2.0 [0.0,3.0]

Remarque : il est possible d'ajouter d'autres résultats quantitatives de nos valeurs tels que la variance ou la valeur maximum... dans ce cas-ci nous pouvons utiliser par exemple le fonction describe() de pandas qui est pratique et simple pour générer des statistiques descriptives de nos valeurs...